In [2]:

import numpy as np
from scipy.ndimage import gaussian_filter

rng = np.random.default_rng(0)
GRID = 60          # 60x60 格点，模拟全球网格
SIGMA = 3.0        # 空间平滑尺度（越大=空间自相关范围越大，越像真实气候场）
N_TRIALS = 200
N_PERM = 200

def smooth_field(grid, sigma, rng):
    raw = rng.normal(size=(grid, grid))
    return gaussian_filter(raw, sigma=sigma, mode='wrap')

def spatial_block_shuffle(field, block, rng):
    n = field.shape[0]
    nb = n // block
    blocks = field[:nb*block, :nb*block].reshape(nb, block, nb, block)
    order_i = rng.permutation(nb)
    order_j = rng.permutation(nb)
    shuffled = blocks[order_i][:, :, order_j]
    return shuffled.transpose(0,1,2,3).reshape(nb*block, nb*block)

naive_fp = 0
block_fp = 0

for trial in range(N_TRIALS):
    # P 和 ET 完全独立生成的两个"平滑"空间场（真null：没有任何因果关系）
    P  = smooth_field(GRID, SIGMA, rng)
    ET = smooth_field(GRID, SIGMA, rng)

    obs_r = abs(np.corrcoef(P.ravel(), ET.ravel())[0, 1])

    # naive: 把 ET 在所有格点间逐点随机打乱（摧毁空间结构）
    naive_null = []
    for _ in range(N_PERM):
        et_shuf = rng.permutation(ET.ravel())
        naive_null.append(abs(np.corrcoef(P.ravel(), et_shuf)[0, 1]))
    p_naive = (np.sum(np.array(naive_null) >= obs_r) + 1) / (N_PERM + 1)

    # spatial block permutation: 按 6x6 的地理小块整体打乱位置，块内空间结构保留
    block_null = []
    for _ in range(N_PERM):
        et_block = spatial_block_shuffle(ET, block=6, rng=rng)
        block_null.append(abs(np.corrcoef(P.ravel(), et_block.ravel())[0, 1]))
    p_block = (np.sum(np.array(block_null) >= obs_r) + 1) / (N_PERM + 1)

    if p_naive <= 0.05:
        naive_fp += 1
    if p_block <= 0.05:
        block_fp += 1

print(f'真实情况：P 和 ET 是两个完全独立生成的空间场（真 null），跑 {N_TRIALS} 组')
print(f'逐点打乱(naive)         误报率（应该≈5%）: {naive_fp/N_TRIALS:.1%}')
print(f'空间分块打乱(block=6x6)  误报率（应该≈5%）: {block_fp/N_TRIALS:.1%}')


真实情况：P 和 ET 是两个完全独立生成的空间场（真 null），跑 200 组
逐点打乱(naive)         误报率（应该≈5%）: 79.5%
空间分块打乱(block=6x6)  误报率（应该≈5%）: 17.5%
